# Backend del clonador de voz (XTTS v2)

1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y elegí **GPU (T4)**.
2. Ejecutá las celdas en orden (o **Entorno de ejecución → Ejecutar todo**). Si Colab pide reiniciar la sesión después de instalar, aceptá y ejecutá desde la segunda celda.
3. La segunda celda descarga el modelo (unos minutos la primera vez) y al final muestra una URL pública que termina en `.gradio.live`. Copiala y pegala en la web.
4. Dejá esta pestaña abierta mientras uses la app. El link dura hasta 72 horas y cambia cada vez que lo ejecutás.

Usalo solo con tu voz o con la de alguien que te dio permiso.

In [ ]:
!pip install -q coqui-tts gradio soundfile

In [ ]:
import os
import re
import subprocess
import tempfile

# Aceptar la licencia de XTTS sin prompt interactivo (necesario en un Space).
os.environ["COQUI_TOS_AGREED"] = "1"

import gradio as gr
import numpy as np
import soundfile as sf
import torch
from TTS.api import TTS

MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"
SAMPLE_RATE = 24000
MAX_CHARS_PER_CHUNK = 200  # XTTS rinde mejor con fragmentos cortos
MAX_TOTAL_CHARS = 5000     # tope de seguridad por pedido
SILENCE_SECONDS = 0.25     # pausa entre fragmentos
MAX_REFERENCES = 8         # XTTS promedia varias muestras: más variedad, mejor parecido
MIN_REFERENCE_SECONDS = 6  # menos que esto rinde mal

LANGUAGES = {
    "Español": "es",
    "English": "en",
    "Português": "pt",
    "Français": "fr",
    "Italiano": "it",
    "Deutsch": "de",
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cargando {MODEL_NAME} en {device}...")
tts = TTS(MODEL_NAME).to(device)
print("Modelo listo.")


def prepare_reference(path: str, workdir: str, idx: int) -> str:
    """Convierte cualquier audio (WhatsApp .opus, .m4a, .mp3, .wav...) a WAV mono 22,05 kHz.

    Además filtra graves molestos (ruido de manos/viento), recorta silencios largos y
    normaliza el volumen, para que todas las muestras tengan un nivel parecido.
    """
    out = os.path.join(workdir, f"ref_{idx}.wav")
    audio_filter = (
        "highpass=f=70,"
        "silenceremove=start_periods=1:start_threshold=-50dB:"
        "stop_periods=-1:stop_duration=0.8:stop_threshold=-50dB,"
        "loudnorm=I=-20:TP=-2"
    )
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error", "-i", path,
        "-vn", "-ac", "1", "-ar", "22050", "-af", audio_filter, out,
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=120)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        raise gr.Error(f"No pude leer el audio «{os.path.basename(path)}». Probá con otro archivo.")
    return out


def split_text(text: str, max_chars: int = MAX_CHARS_PER_CHUNK) -> list[str]:
    """Divide el texto en fragmentos por oraciones, sin pasarse de max_chars."""
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?¡¿;:])\s+", text)

    chunks: list[str] = []
    current = ""
    for sentence in sentences:
        if not sentence:
            continue
        # Oración demasiado larga: cortarla por comas y, si hace falta, por palabras.
        if len(sentence) > max_chars:
            parts = re.split(r"(?<=,)\s+", sentence)
            pieces: list[str] = []
            for part in parts:
                while len(part) > max_chars:
                    cut = part.rfind(" ", 0, max_chars)
                    cut = cut if cut > 0 else max_chars
                    pieces.append(part[:cut].strip())
                    part = part[cut:].strip()
                if part:
                    pieces.append(part)
        else:
            pieces = [sentence]

        for piece in pieces:
            if len(current) + len(piece) + 1 <= max_chars:
                current = f"{current} {piece}".strip()
            else:
                if current:
                    chunks.append(current)
                current = piece
    if current:
        chunks.append(current)
    return chunks


def clone(ref_audio, text: str, language: str):
    # El frontend manda una lista de archivos; Gradio puede entregar un solo path o una lista.
    if isinstance(ref_audio, (str, os.PathLike)):
        ref_audio = [ref_audio]
    refs = [str(p) for p in (ref_audio or [])]
    if not refs:
        raise gr.Error("Subí al menos un audio de referencia (mejor 3 a 5 audios limpios de la misma persona).")
    if len(refs) > MAX_REFERENCES:
        raise gr.Error(f"Subiste {len(refs)} audios. El máximo es {MAX_REFERENCES}.")
    if not text or not text.strip():
        raise gr.Error("Escribí el texto que querés generar.")
    if len(text) > MAX_TOTAL_CHARS:
        raise gr.Error(f"El texto es muy largo ({len(text)} caracteres). El máximo es {MAX_TOTAL_CHARS}.")

    lang_code = LANGUAGES.get(language, language)

    with tempfile.TemporaryDirectory() as workdir:
        wavs = [prepare_reference(p, workdir, i) for i, p in enumerate(refs)]
        total = sum(sf.info(w).duration for w in wavs)
        if total < MIN_REFERENCE_SECONDS:
            raise gr.Error(
                f"Entre todos los audios suman {total:.1f} segundos de voz (después de sacar silencios). "
                f"Necesito al menos {MIN_REFERENCE_SECONDS}; lo ideal es 20 a 30."
            )

        chunks = split_text(text)
        silence = np.zeros(int(SAMPLE_RATE * SILENCE_SECONDS), dtype=np.float32)

        pieces: list[np.ndarray] = []
        for i, chunk in enumerate(chunks):
            wav = tts.tts(text=chunk, speaker_wav=wavs, language=lang_code)
            pieces.append(np.asarray(wav, dtype=np.float32))
            if i < len(chunks) - 1:
                pieces.append(silence)

    audio = np.concatenate(pieces)
    out_path = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    sf.write(out_path, audio, SAMPLE_RATE)
    return out_path


with gr.Blocks(title="Clonador de voz") as demo:
    gr.Markdown(
        "# Clonador de voz (XTTS v2)\n"
        "Subí uno o varios audios de referencia (WhatsApp sirve), escribí el texto y generá el audio. "
        "Usalo solo con tu propia voz o con la de alguien que te dio permiso."
    )
    with gr.Row():
        with gr.Column():
            ref = gr.File(label="Audios de referencia", file_count="multiple", type="filepath")
            txt = gr.Textbox(label="Texto", lines=6)
            lang = gr.Dropdown(list(LANGUAGES.keys()), value="Español", label="Idioma")
            btn = gr.Button("Generar", variant="primary")
        with gr.Column():
            out = gr.Audio(label="Resultado", type="filepath")

    # api_name="clone" es el endpoint que usa el frontend de Vercel.
    btn.click(clone, inputs=[ref, txt, lang], outputs=out, api_name="clone")

# share=True genera un link público https://xxxx.gradio.live (necesario en Colab).
demo.queue(max_size=10).launch(share=True)
